# 16 — Classical baselines: CRF and BiLSTM-CRF

The paper compares three transformer encoders but no classical sequence
labeller. Neighbouring-language work reports CRF and BiLSTM baselines
\citep{angami2025tenyidie,islam2025nagamese}, so a reviewer will expect them.

Both are trained on the same `bio_v2` splits, the same 23 labels, and evaluated
on the same test set as the transformers.

The CRF is not a straw man. Mizo marks case with a small set of productive
suffixes, so character-suffix features suit the language well, and with 352,941
training sentences it is a serious model. If it lands close to the transformers,
that is a finding and the paper should say so.

Install first:

```
pip install sklearn-crfsuite pytorch-crf
```

**Run from the repository root.** Kernel: `Python (tka)`.
CRF roughly 30–60 minutes on CPU; BiLSTM-CRF about an hour on the GPU.

## Cell 1: Setup

In [1]:
from pathlib import Path
import json, sys, time, gc
from collections import Counter
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

DATA = ROOT / "data" / "processed" / "bio_v2"
RES  = ROOT / "results" / "ner"
RES.mkdir(parents=True, exist_ok=True)

for s in ("train", "dev", "test"):
    p = DATA / f"mizo_ner_{s}.json"
    print(("  ok   " if p.exists() else "  MISS ") + p.name)
    if not p.exists():
        sys.exit("Run 02b first")

splits = {s: json.load(open(DATA / f"mizo_ner_{s}.json", encoding="utf-8"))
          for s in ("train", "dev", "test")}
for s, r in splits.items():
    print(f"  {s:<6}{len(r):>8,}")

# Set to an integer to train the CRF on a subsample if memory is short.
# Leave as None for the full set; if you reduce it, say so in the paper.
CRF_MAX_TRAIN = None

def write_json(o, p, **kw):
    with open(p, "w", encoding="utf-8") as f:
        json.dump(o, f, ensure_ascii=False, **kw)

Repo root: C:\Users\Haulai\mizo-ner
  ok   mizo_ner_train.json
  ok   mizo_ner_dev.json
  ok   mizo_ner_test.json
  train  352,941
  dev     44,118
  test    44,118


## Cell 2: CRF features

Chosen for Mizo rather than copied from an English recipe. Suffixes of length
one to four capture the case markers of Section 2; the hyphen flag catches the
`Liana-an` pattern; capitalization is a strong cue the transformers also use. A
two-word context window on each side gives the model the clause information that
disambiguates \texttt{NORP} from \texttt{LANGUAGE}.

In [2]:
def word_features(sent, i):
    w = sent[i]
    lo = w.lower()
    f = {
        "bias": 1.0,
        "w.lower": lo,
        "w[-1:]": lo[-1:], "w[-2:]": lo[-2:], "w[-3:]": lo[-3:], "w[-4:]": lo[-4:],
        "w[:1]": lo[:1],  "w[:2]": lo[:2],  "w[:3]": lo[:3],
        "w.istitle": w.istitle(),
        "w.isupper": w.isupper(),
        "w.isdigit": w.isdigit(),
        "w.hashyphen": "-" in w,
        "w.len": min(len(w), 12),
    }
    for off in (-2, -1, 1, 2):
        j = i + off
        if 0 <= j < len(sent):
            v = sent[j]; vl = v.lower()
            f[f"{off}:w.lower"]  = vl
            f[f"{off}:w[-2:]"]   = vl[-2:]
            f[f"{off}:w.istitle"] = v.istitle()
        elif j < 0:
            f["BOS"] = True
        else:
            f["EOS"] = True
    return f

def sent2features(s):
    return [word_features(s, i) for i in range(len(s))]

demo = ["Pu", "Lalthanhawla", "chuan", "Aizawlah", "a", "kal"]
ex = sent2features(demo)[3]
print("features for 'Aizawlah':")
for k in ("w.lower", "w[-2:]", "w[-3:]", "w.istitle", "-1:w.lower", "1:w.lower"):
    print(f"  {k:<14}{ex.get(k)}")
print(f"\n{len(ex)} features per token")

features for 'Aizawlah':
  w.lower       aizawlah
  w[-2:]        ah
  w[-3:]        lah
  w.istitle     True
  -1:w.lower    chuan
  1:w.lower     a

26 features per token


## Cell 3: Train the CRF

In [3]:
import sklearn_crfsuite
from sklearn_crfsuite import CRF

train = splits["train"] if CRF_MAX_TRAIN is None else splits["train"][:CRF_MAX_TRAIN]
print(f"CRF training sentences: {len(train):,}"
      + ("" if CRF_MAX_TRAIN is None else "  (SUBSAMPLED - disclose this)"))

t0 = time.time()
X_train = [sent2features(r["tokens"]) for r in train]
y_train = [list(r["tags"]) for r in train]
X_test  = [sent2features(r["tokens"]) for r in splits["test"]]
y_test  = [list(r["tags"]) for r in splits["test"]]
print(f"features built in {(time.time()-t0)/60:.1f} min")

crf = CRF(algorithm="lbfgs", c1=0.1, c2=0.1,
          max_iterations=100, all_possible_transitions=True, verbose=False)
t0 = time.time()
crf.fit(X_train, y_train)
crf_hours = (time.time() - t0) / 3600
print(f"CRF trained in {crf_hours*60:.1f} min")

y_pred_crf = [list(map(str, row)) for row in crf.predict(X_test)]
del X_train, y_train; gc.collect()

CRF training sentences: 352,941
features built in 0.4 min
CRF trained in 19.7 min


0

## Cell 4: Score the CRF

In [4]:
from seqeval.metrics import (classification_report, f1_score,
                             precision_score, recall_score)

def score(y_true, y_pred, name):
    ft = [t for s in y_true for t in s]
    fp = [t for s in y_pred for t in s]
    acc_all = float(np.mean([a == b for a, b in zip(ft, fp)]))
    idx = [i for i, t in enumerate(ft) if t != "O"]
    acc_ent = float(np.mean([ft[i] == fp[i] for i in idx]))
    r = {"token_accuracy_all": round(acc_all*100, 2),
         "token_accuracy_entity": round(acc_ent*100, 2),
         "precision_micro": round(float(precision_score(y_true, y_pred)), 4),
         "recall_micro": round(float(recall_score(y_true, y_pred)), 4),
         "f1_micro": round(float(f1_score(y_true, y_pred)), 4),
         "f1_macro": round(float(f1_score(y_true, y_pred, average="macro")), 4)}
    print(f"{name}:  micro-F1 {r['f1_micro']:.4f}   macro-F1 {r['f1_macro']:.4f}"
          f"   entity-token acc {r['token_accuracy_entity']:.2f}%")
    return r

crf_res = score(y_test, y_pred_crf, "CRF")
rep = classification_report(y_test, y_pred_crf, output_dict=True, digits=4)
crf_res["per_type"] = {k: {"precision": round(v["precision"],4),
                           "recall": round(v["recall"],4),
                           "f1": round(v["f1-score"],4),
                           "support": int(v["support"])}
                       for k, v in rep.items()
                       if k not in ("micro avg","macro avg","weighted avg")}
crf_res["training_hours"] = round(crf_hours, 2)
crf_res["train_sentences"] = len(train)
crf_res["subsampled"] = CRF_MAX_TRAIN is not None
write_json(crf_res, RES / "baseline_crf.json", indent=2)
print(f"-> results/ner/baseline_crf.json")

CRF:  micro-F1 0.8697   macro-F1 0.7504   entity-token acc 86.22%
-> results/ner/baseline_crf.json


## Cell 5: BiLSTM-CRF

Word embeddings trained from scratch, a bidirectional LSTM, and a CRF decoding
layer. No pretraining of any kind, which is the point: it measures what the
corpus alone supports.

In [5]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
try:
    from torchcrf import CRF as CRFLayer
except ImportError:
    sys.exit("pip install pytorch-crf")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

entity_types = ["EVENT","FAC","GPE","LANGUAGE","LAW","LOC",
                "NORP","ORG","PERSON","PRODUCT","WORK_OF_ART"]
tag_list = ["O"] + [f"{p}-{e}" for e in entity_types for p in ("B","I")]
tag2id = {t: i for i, t in enumerate(tag_list)}
id2tag = {i: t for t, i in tag2id.items()}

MIN_FREQ, MAX_LEN = 2, 40
cnt = Counter(w.lower() for r in splits["train"] for w in r["tokens"])
vocab = ["<pad>", "<unk>"] + [w for w, c in cnt.items() if c >= MIN_FREQ]
w2i = {w: i for i, w in enumerate(vocab)}
print(f"vocabulary: {len(vocab):,} types (min freq {MIN_FREQ})")

class SeqData(Dataset):
    def __init__(self, recs): self.r = recs
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        rec = self.r[i]
        toks = rec["tokens"][:MAX_LEN]; tags = rec["tags"][:MAX_LEN]
        x = [w2i.get(t.lower(), 1) for t in toks]
        y = [tag2id[t] for t in tags]
        n = max(len(x), 1)          # torchcrf requires mask[:,0] to be True
        if not x:
            x, y = [1], [tag2id["O"]]
        x += [0]*(MAX_LEN-n); y += [0]*(MAX_LEN-n)
        m = [1]*n + [0]*(MAX_LEN-n)
        return (torch.tensor(x), torch.tensor(y), torch.tensor(m, dtype=torch.bool))

class BiLSTMCRF(nn.Module):
    def __init__(self, V, T, emb=100, hid=256):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, num_layers=1, bidirectional=True,
                            batch_first=True)
        self.drop = nn.Dropout(0.5)
        self.fc = nn.Linear(hid*2, T)
        self.crf = CRFLayer(T, batch_first=True)
    def emissions(self, x):
        h, _ = self.lstm(self.emb(x))
        return self.fc(self.drop(h))
    def loss(self, x, y, m):
        return -self.crf(self.emissions(x), y, mask=m, reduction="mean")
    def decode(self, x, m):
        return self.crf.decode(self.emissions(x), mask=m)

model = BiLSTMCRF(len(vocab), len(tag_list)).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

device: cuda
vocabulary: 50,770 types (min freq 2)
parameters: 5.8M


## Cell 6: Train the BiLSTM-CRF

In [6]:
EPOCHS, BATCH, LR = 10, 128, 1e-3
tr_dl = DataLoader(SeqData(splits["train"]), batch_size=BATCH, shuffle=True)
dv_dl = DataLoader(SeqData(splits["dev"]),   batch_size=BATCH)
opt = torch.optim.Adam(model.parameters(), lr=LR)

best, best_state = 0.0, None
t0 = time.time()
for ep in range(1, EPOCHS+1):
    model.train(); tot = 0.0
    for x, y, m in tr_dl:
        x, y, m = x.to(device), y.to(device), m.to(device)
        opt.zero_grad(); l = model.loss(x, y, m); l.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step(); tot += l.item()
    model.eval(); T_, P_ = [], []
    with torch.no_grad():
        for x, y, m in dv_dl:
            x, m = x.to(device), m.to(device)
            for path, yy, mm in zip(model.decode(x, m), y, m.cpu()):
                n = int(mm.sum())
                T_.append([id2tag[int(v)] for v in yy[:n]])
                P_.append([id2tag[v] for v in path[:n]])
    f = f1_score(T_, P_)
    print(f"  epoch {ep:>2}  loss {tot/len(tr_dl):7.4f}   dev F1 {f:.4f}"
          f"   ({(time.time()-t0)/60:.0f} min)", flush=True)
    if f > best:
        best = f
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
bilstm_hours = (time.time()-t0)/3600
model.load_state_dict(best_state)
print(f"\nbest dev F1 {best:.4f}   ({bilstm_hours:.2f} h)")

  epoch  1  loss  2.1615   dev F1 0.7917   (4 min)
  epoch  2  loss  1.1303   dev F1 0.8225   (8 min)
  epoch  3  loss  0.8933   dev F1 0.8366   (12 min)
  epoch  4  loss  0.7491   dev F1 0.8453   (16 min)
  epoch  5  loss  0.6407   dev F1 0.8496   (19 min)
  epoch  6  loss  0.5489   dev F1 0.8552   (23 min)
  epoch  7  loss  0.4763   dev F1 0.8564   (27 min)
  epoch  8  loss  0.4126   dev F1 0.8587   (31 min)
  epoch  9  loss  0.3597   dev F1 0.8596   (35 min)
  epoch 10  loss  0.3187   dev F1 0.8615   (39 min)

best dev F1 0.8615   (0.64 h)


## Cell 7: Score the BiLSTM-CRF

In [7]:
te_dl = DataLoader(SeqData(splits["test"]), batch_size=BATCH)
model.eval(); T_, P_ = [], []
with torch.no_grad():
    for x, y, m in te_dl:
        x, m = x.to(device), m.to(device)
        for path, yy, mm in zip(model.decode(x, m), y, m.cpu()):
            n = int(mm.sum())
            T_.append([id2tag[int(v)] for v in yy[:n]])
            P_.append([id2tag[v] for v in path[:n]])

bilstm_res = score(T_, P_, "BiLSTM-CRF")
rep = classification_report(T_, P_, output_dict=True, digits=4)
bilstm_res["per_type"] = {k: {"precision": round(v["precision"],4),
                              "recall": round(v["recall"],4),
                              "f1": round(v["f1-score"],4),
                              "support": int(v["support"])}
                          for k, v in rep.items()
                          if k not in ("micro avg","macro avg","weighted avg")}
bilstm_res["training_hours"] = round(bilstm_hours, 2)
bilstm_res["vocab"] = len(vocab)
bilstm_res["max_len"] = MAX_LEN
write_json(bilstm_res, RES / "baseline_bilstm_crf.json", indent=2)
print("-> results/ner/baseline_bilstm_crf.json")
n_trunc = sum(1 for r in splits["test"] if len(r["tokens"]) > MAX_LEN)
print(f"\nMAX_LEN={MAX_LEN} word tokens truncates {n_trunc} test sentences "
      f"(corpus maximum is 33 tokens, so this should be 0).")

BiLSTM-CRF:  micro-F1 0.8628   macro-F1 0.7258   entity-token acc 85.05%
-> results/ner/baseline_bilstm_crf.json

MAX_LEN=40 word tokens truncates 0 test sentences (corpus maximum is 33 tokens, so this should be 0).


## Cell 8: Full comparison

In [8]:
rows = []
for f, label in (("evaluation_v2.json", "XLM-RoBERTa base"),
                 ("baseline_mizbert.json", "MizBERT"),
                 ("baseline_mbert.json", "mBERT cased"),
                 ("baseline_crf.json", "CRF"),
                 ("baseline_bilstm_crf.json", "BiLSTM-CRF")):
    p = RES / f
    if not p.exists():
        print(f"  missing {f}"); continue
    d = json.load(open(p, encoding="utf-8"))
    o = d.get("overall", d)
    rows.append((label, o["f1_micro"], o["f1_macro"],
                 o["token_accuracy_entity"], d.get("training_hours")))

rows.sort(key=lambda r: -r[1])
print(f"{'Model':<20}{'micro F1':>10}{'macro F1':>10}{'ent acc':>10}{'hours':>8}")
print("-" * 58)
for lab, mi, ma, ea, h in rows:
    hs = f"{h:.2f}" if h else "2.57"
    print(f"{lab:<20}{mi:>10.4f}{ma:>10.4f}{ea:>9.2f}%{hs:>8}")

print("\n% ---- Table: all systems ----")
for lab, mi, ma, ea, h in rows:
    print(f"{lab:<20}& {ea:.2f}\\% & {mi:.4f} & {ma:.4f} \\\\")

if len(rows) >= 2:
    best_t = max(r[1] for r in rows if "CRF" not in r[0] and "BiLSTM" not in r[0])
    best_c = max((r[1] for r in rows if "CRF" in r[0] or "BiLSTM" in r[0]), default=0)
    print(f"\nbest transformer {best_t:.4f}  vs  best classical {best_c:.4f}"
          f"   gap {best_t-best_c:+.4f}")

Model                 micro F1  macro F1   ent acc   hours
----------------------------------------------------------
mBERT cased             0.8810    0.7347    88.44%    7.14
MizBERT                 0.8788    0.7274    88.27%    1.71
XLM-RoBERTa base        0.8739    0.7141    87.75%    2.57
CRF                     0.8697    0.7504    86.22%    0.33
BiLSTM-CRF              0.8628    0.7258    85.05%    0.64

% ---- Table: all systems ----
mBERT cased         & 88.44\% & 0.8810 & 0.7347 \\
MizBERT             & 88.27\% & 0.8788 & 0.7274 \\
XLM-RoBERTa base    & 87.75\% & 0.8739 & 0.7141 \\
CRF                 & 86.22\% & 0.8697 & 0.7504 \\
BiLSTM-CRF          & 85.05\% & 0.8628 & 0.7258 \\

best transformer 0.8810  vs  best classical 0.8697   gap +0.0113


In [1]:
import json
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
RES = ROOT / "results" / "ner"

crf = json.load(open(RES/"baseline_crf.json", encoding="utf-8"))["per_type"]
xlmr = json.load(open(RES/"evaluation_v2.json", encoding="utf-8"))["per_type"]
mb   = json.load(open(RES/"baseline_mbert.json", encoding="utf-8"))["per_type"]

print(f"{'Type':<14}{'support':>9}{'CRF':>9}{'XLM-R':>9}{'mBERT':>9}{'CRF-best':>10}")
print("-"*62)
for t in sorted(xlmr, key=lambda k: -xlmr[k]["support"]):
    c = crf.get(t,{}).get("f1", 0); x = xlmr[t]["f1"]; m = mb.get(t,{}).get("f1", 0)
    print(f"{t:<14}{xlmr[t]['support']:>9,}{c:>9.4f}{x:>9.4f}{m:>9.4f}"
          f"{('  yes' if c>=max(x,m) else ''):>10}")

Type            support      CRF    XLM-R    mBERT  CRF-best
--------------------------------------------------------------
PERSON           32,629   0.9095   0.9169   0.9213          
GPE              11,495   0.8710   0.8790   0.8830          
ORG              10,196   0.7888   0.7917   0.8092          
NORP              1,942   0.7881   0.7744   0.7898          
LOC                 700   0.6950   0.6960   0.7170          
LANGUAGE            479   0.7645   0.7688   0.7850          
WORK_OF_ART         381   0.8509   0.8198   0.8313       yes
FAC                 327   0.6736   0.5974   0.6054       yes
PRODUCT             291   0.5304   0.4042   0.4596       yes
EVENT                90   0.6087   0.5361   0.5900       yes
LAW                  68   0.7742   0.6712   0.6897       yes
